## Polygonization Exercise

In [ ]:
from pygel3d import hmesh, jupyter_display as jd
jd.set_export_mode(True)
import numpy as np
from numpy.linalg import norm
from math import exp,floor
array = np.array
        
cube_faces = np.array([[[0,-0.5,-0.5],[0,0.5,-0.5],[0,0.5,0.5],[0,-0.5,0.5]],
    [[0, 0.5,-0.5],[0,-0.5,-0.5],[0,-0.5,0.5],[0,0.5,0.5]],
    [[ 0.5,0, -0.5],[-0.5,0, -0.5],[-0.5,0, 0.5],[0.5,0, 0.5]],
    [[-0.5,0, -0.5],[0.5,0, -0.5],[0.5,0, 0.5],[-0.5,0, 0.5]],
    [[-0.5,-0.5,0],[0.5,-0.5,0],[0.5,0.5,0],[-0.5,0.5,0]],
    [[ 0.5,-0.5,0],[-0.5,-0.5,0],[-0.5,0.5,0],[0.5,0.5,0]]])

neighbours = np.array([[1.,0.,0.],[-1.,0.,0.],[0.,1.,0.],[0.,-1.,0.],[0.,0.,1.],[0.,0.,-1.]])

In [ ]:
class SphereVolume:
    def __init__(self,_center,_radius):
        self.center = _center
        self.radius = _radius
    def eval(self,p):
        return norm(p-self.center) - self.radius
    def grad(self,p):
        v = p - self.center
        return v / norm(v)

### The XForm Class

The class below is used to transform between the 3D grid and the domain in which your output should live. For instance, say you have a unit sphere represented as a distance field. Perhaps you want the unit sphere to exist in the domain [-1,1]^3 but use a grid of 16x16x16 polygonization cells. In this case the point [0,0,0] in the voxel grid should map to [-1,-1,-1] in the domain and [16,16,16] should map to [1,1,1]. The `XForm` class takes care of this mapping and the inverse.


In [ ]:
class XForm:
    def __init__(self,Plow,Phigh,dims):
        self.dims = np.array(dims)
        self.Plow = np.array(Plow)
        self.Phigh = np.array(Phigh)
        self.scaling = (self.Phigh - self.Plow) / self.dims
    def map(self,pi):
        return self.scaling * np.array(pi) + self.Plow

### Part 1: Finish the polygonizer
You need to find the cube faces that should be generated and also project vertices back onto the isosurface

In [ ]:
def smooth(m):
    pos = m.positions()
    new_pos = np.zeros(pos.shape)
    for i in range(0, 5):
        for curr_idx in m.vertices():
            neigbors = list(m.circulate_vertex(curr_idx, mode='v'))
            new_pos[curr_idx] = np.mean(pos[neigbors], axis=0)
        pos[:] = new_pos


def polygonize_volume(vol, xform, tau):
    m = hmesh.Manifold()
    
    for pi in np.ndindex(tuple(xform.dims)):
        pi = np.array(pi, dtype=float)
        if vol.eval(xform.map(pi)) < tau:
            for i in range(len(cube_faces)):
                face = cube_faces[i]
                
                neighbor = neighbours[i]
                nbr = pi + neighbor
                if vol.eval(xform.map(nbr)) > tau:
                    m.add_face(pi + 0.5 * neighbor + face)

    hmesh.stitch(m)
    m.cleanup()
    pos = m.positions()
    smooth(m)

    
    for v in m.vertices():
        p = xform.map(pos[v])
        # Newton step onto f(p) = tau. Exact in one step for a distance field;
        # a few iterations pull a general scalar field onto the level set.
        for _ in range(5):
            g = vol.grad(p)
            p = p - (vol.eval(p) - tau) * g / np.dot(g, g)

        pos[v] = (p - xform.Plow) / xform.scaling
    


    
    return m

In [ ]:
vol = SphereVolume([0.,0,0.], 1.4)

xform = XForm([-1.5,-1.5,-1.5],[1.5,1.5,1.5],[16,16,16])

m = polygonize_volume(vol, xform, 0)
jd.display(m)

### Part 2: Create Gaussian and Gradient of Gaussian functions

In [ ]:
### Exercise: Create functions that return value of 3D Gaussian and gradient. 
ssq = 0.1**2

def Gaussian(p0, p):
    diff = np.array(p) - np.array(p0)
    return np.exp(-np.dot(diff, diff) / (2 * ssq))

def GaussianGrad(p0, p):
    diff = np.array(p) - np.array(p0)
    value = Gaussian(p0, p)
    return -diff / ssq * value

In [ ]:
class Blobs:
    def __init__(self,centers):
        self.centers = centers
    def eval(self,pp):
        p = np.divmod(pp,1)[1]
        return sum([Gaussian(c, p) for c in self.centers])
    def grad(self,pp):
        p = np.divmod(pp,1)[1]
        return sum([GaussianGrad(c, p) for c in self.centers])

### Part 3: play with Blobs
Finally, when you see the blobby cube, try tweaking it to model something different.

In [ ]:
cntrs = [array([0.2,0.2,0.3]) + array([0.5*p[0],0.5*p[1],0.31*p[2]]) for p in np.ndindex((2,2,2))]
vol = Blobs(cntrs)

xform = XForm([0,0,0],[1,1,1],[16,16,16])
m = polygonize_volume(vol, xform, 0.35)
hmesh.flip_orientation(m)
jd.display(m)

## Questions

- What is a distance field and how does it relate to a scalar field?

A distance field is a scalar field whose value is the signed distance to the surface: negative inside, zero on the surface, positive outside.

- Explain the difference between primal and dual isocontouring methods.

Primal methods place a vertex on each grid edge that crosses the isovalue and connect those vertices inside the cell. Dual methods give one vertex to each cut cell and connect the vertices of the cells that share a crossing edge.

- Explain the method for projecting points onto the isocontour.

Move the point along the gradient, `p = p - (f(p) - tau) * grad / ||grad||^2`, and repeat until `f(p) = tau`. One step is exact for a distance field.
